# Module 03: Autoencoders — AE, VAE, VQ-VAE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/03-autoencoders/notebook.ipynb)

**GPU recommended:** Yes (VAE training on MNIST is faster with GPU, ~5 min on CPU).

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": False,
    "figure.dpi": 100,
})

## Load MNIST

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(train_dataset[i][0].squeeze(), cmap="gray")
    ax.set_title(str(train_dataset[i][1]), fontsize=9)
    ax.axis("off")
fig.suptitle("Sample MNIST digits", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 1: Vanilla Autoencoder

A vanilla autoencoder learns to compress data into a low-dimensional latent space
(encoder) and reconstruct it back (decoder). The loss is simply the reconstruction
error (MSE). There is no explicit regularization on the latent space.

In [ ]:
class VanillaEncoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 128)
        self.fc3 = nn.Linear(128, latent_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class VanillaDecoder(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=256, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 128)
        self.fc2 = nn.Linear(128, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        z = F.relu(self.fc1(z))
        z = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(z))


class VanillaAutoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super().__init__()
        self.encoder = VanillaEncoder(input_dim, hidden_dim, latent_dim)
        self.decoder = VanillaDecoder(latent_dim, hidden_dim, input_dim)

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

In [ ]:
def train_vanilla_ae(model, train_loader, epochs=20, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_imgs, _ in train_loader:
            batch_imgs = batch_imgs.view(-1, 784).to(device)
            x_recon, _ = model(batch_imgs)
            loss = F.mse_loss(x_recon, batch_imgs)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch_imgs.size(0)

        avg_loss = epoch_loss / len(train_loader.dataset)
        losses.append(avg_loss)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | MSE Loss: {avg_loss:.6f}")

    return losses

In [ ]:
ae_model = VanillaAutoencoder(latent_dim=2).to(device)
print(f"Vanilla AE parameters: {sum(p.numel() for p in ae_model.parameters()):,}")
print("\nTraining Vanilla Autoencoder...")
ae_losses = train_vanilla_ae(ae_model, train_loader, epochs=20)

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(ae_losses, linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Vanilla AE Training Loss")
plt.tight_layout()
plt.show()

### Vanilla AE Reconstructions

In [ ]:
def show_reconstructions(model, data_loader, n=10, title="Reconstructions"):
    model.eval()
    batch_imgs, _ = next(iter(data_loader))
    batch_imgs = batch_imgs[:n].to(device)
    with torch.no_grad():
        x_flat = batch_imgs.view(-1, 784)
        recon, _ = model(x_flat)
        recon = recon.view(-1, 1, 28, 28)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 2.8))
    for i in range(n):
        axes[0, i].imshow(batch_imgs[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Original", fontsize=10)
    axes[1, 0].set_ylabel("Recon", fontsize=10)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


show_reconstructions(ae_model, test_loader, title="Vanilla AE Reconstructions")

---
## Part 2: Latent Space Visualization (Vanilla AE)

With a 2D bottleneck we can directly plot the latent codes. Because the vanilla AE
has no regularization on its latent space, we expect to see separated clusters but
with gaps and irregular structure between them.

In [ ]:
def encode_dataset(model, data_loader):
    model.eval()
    all_z = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs = imgs.view(-1, 784).to(device)
            _, z = model(imgs)
            all_z.append(z.cpu())
            all_labels.append(labels)
    return torch.cat(all_z), torch.cat(all_labels)


ae_z, ae_labels = encode_dataset(ae_model, test_loader)

In [ ]:
def plot_latent_space(z, labels, title="Latent Space"):
    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(
        z[:, 0].numpy(), z[:, 1].numpy(),
        c=labels.numpy(), cmap="tab10", s=2, alpha=0.6
    )
    cbar = plt.colorbar(scatter, ticks=range(10))
    cbar.set_label("Digit")
    plt.xlabel("z_0")
    plt.ylabel("z_1")
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_latent_space(ae_z, ae_labels, title="Vanilla AE Latent Space (2D)")

Notice how the vanilla AE latent space has clusters for each digit, but the
distribution is irregular: there are gaps, overlapping regions, and no guarantee
that sampling a random point will produce a meaningful digit.

---
## Part 3: Variational Autoencoder (VAE)

The VAE adds probabilistic structure to the latent space. Instead of encoding
to a single point, the encoder outputs the mean and log-variance of a Gaussian.
We sample from this Gaussian using the **reparameterization trick** (so gradients
can flow through the sampling step).

The loss is the **ELBO** (Evidence Lower Bound):
- **Reconstruction term:** how well the decoder reconstructs the input
- **KL divergence term:** regularizes the latent distribution toward N(0, I)

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 128)
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=256, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 128)
        self.fc2 = nn.Linear(128, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))
        h = F.relu(self.fc2(h))
        return torch.sigmoid(self.fc3(h))


class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super().__init__()
        self.encoder = VAEEncoder(input_dim, hidden_dim, latent_dim)
        self.decoder = VAEDecoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        """Sample z = mu + sigma * epsilon, where epsilon ~ N(0, I)."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return x_recon, mu, logvar, z

In [ ]:
def vae_loss_fn(x_recon, x, mu, logvar):
    """ELBO = Reconstruction (BCE) + KL divergence."""
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction="sum")
    # KL(q(z|x) || p(z)) where p(z) = N(0, I)
    # = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss, recon_loss, kl_loss

In [ ]:
def train_vae(model, train_loader, epochs=20, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    history = {"total": [], "recon": [], "kl": []}

    for epoch in range(epochs):
        total_loss = 0.0
        total_recon = 0.0
        total_kl = 0.0

        for batch_imgs, _ in train_loader:
            batch_imgs = batch_imgs.view(-1, 784).to(device)
            x_recon, mu, logvar, _ = model(batch_imgs)
            loss, recon, kl = vae_loss_fn(x_recon, batch_imgs, mu, logvar)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon.item()
            total_kl += kl.item()

        n = len(train_loader.dataset)
        history["total"].append(total_loss / n)
        history["recon"].append(total_recon / n)
        history["kl"].append(total_kl / n)

        if (epoch + 1) % 5 == 0:
            print(
                f"  Epoch {epoch+1}/{epochs} | "
                f"Total: {history['total'][-1]:.2f} | "
                f"Recon: {history['recon'][-1]:.2f} | "
                f"KL: {history['kl'][-1]:.2f}"
            )

    return history

In [ ]:
vae_model = VAE(latent_dim=2).to(device)
print(f"VAE parameters: {sum(p.numel() for p in vae_model.parameters()):,}")
print("\nTraining VAE...")
vae_history = train_vae(vae_model, train_loader, epochs=20)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(vae_history["total"], linewidth=1.5)
axes[0].set_title("Total (ELBO)")
axes[1].plot(vae_history["recon"], linewidth=1.5, color="tab:orange")
axes[1].set_title("Reconstruction (BCE)")
axes[2].plot(vae_history["kl"], linewidth=1.5, color="tab:green")
axes[2].set_title("KL Divergence")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (per sample)")
fig.suptitle("VAE Training Losses", fontsize=12)
plt.tight_layout()
plt.show()

### VAE Reconstructions

In [ ]:
def show_vae_reconstructions(model, data_loader, n=10, title="VAE Reconstructions"):
    model.eval()
    batch_imgs, _ = next(iter(data_loader))
    batch_imgs = batch_imgs[:n].to(device)
    with torch.no_grad():
        x_flat = batch_imgs.view(-1, 784)
        recon, _, _, _ = model(x_flat)
        recon = recon.view(-1, 1, 28, 28)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 2.8))
    for i in range(n):
        axes[0, i].imshow(batch_imgs[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Original", fontsize=10)
    axes[1, 0].set_ylabel("Recon", fontsize=10)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


show_vae_reconstructions(vae_model, test_loader)

---
## Part 4: VAE Latent Space — Continuity and Completeness

The KL regularization encourages the VAE latent space to be:
- **Continuous:** nearby points in latent space decode to similar images
- **Complete:** every point sampled from N(0, I) decodes to a plausible digit

We visualize this in two ways: a scatter plot of encoded test digits, and
a grid of decoded samples spanning the latent space.

In [ ]:
def encode_dataset_vae(model, data_loader):
    model.eval()
    all_mu = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs = imgs.view(-1, 784).to(device)
            mu, _ = model.encoder(imgs)
            all_mu.append(mu.cpu())
            all_labels.append(labels)
    return torch.cat(all_mu), torch.cat(all_labels)


vae_z, vae_labels = encode_dataset_vae(vae_model, test_loader)
plot_latent_space(vae_z, vae_labels, title="VAE Latent Space (2D)")

### Grid Sampling: Generating New Digits from the Latent Space

We sample a uniform grid in 2D latent space (using the inverse CDF of the
standard normal to map from quantiles to z-values) and decode each point.

In [ ]:
def plot_latent_grid(model, n=20, z_range=(-3, 3), figsize=(10, 10)):
    model.eval()
    grid_x = np.linspace(z_range[0], z_range[1], n)
    grid_y = np.linspace(z_range[1], z_range[0], n)  # flip y for image coords

    canvas = np.zeros((28 * n, 28 * n))

    with torch.no_grad():
        for i, yi in enumerate(grid_y):
            for j, xi in enumerate(grid_x):
                z = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
                decoded = model.decoder(z).cpu().numpy().reshape(28, 28)
                canvas[i * 28 : (i + 1) * 28, j * 28 : (j + 1) * 28] = decoded

    plt.figure(figsize=figsize)
    plt.imshow(canvas, cmap="gray")
    plt.title("VAE Latent Space Grid Sampling", fontsize=14)
    plt.xlabel(f"z_0 ({z_range[0]} to {z_range[1]})")
    plt.ylabel(f"z_1 ({z_range[0]} to {z_range[1]})")
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()


plot_latent_grid(vae_model, n=20)

The grid shows smooth transitions between digit classes, demonstrating that
the VAE latent space is both continuous and complete.

---
## Part 5: Latent Interpolation Between Two Digits

We encode two different digits and linearly interpolate between their
latent representations to see a smooth morphing.

In [ ]:
def find_digit(dataset, target_digit, index=0):
    """Return the index-th occurrence of target_digit in dataset."""
    count = 0
    for i in range(len(dataset)):
        if dataset[i][1] == target_digit:
            if count == index:
                return dataset[i][0]
            count += 1
    return None


def interpolate_vae(model, img1, img2, steps=10):
    model.eval()
    with torch.no_grad():
        x1 = img1.view(1, 784).to(device)
        x2 = img2.view(1, 784).to(device)
        mu1, _ = model.encoder(x1)
        mu2, _ = model.encoder(x2)

        alphas = np.linspace(0, 1, steps)
        decoded_imgs = []
        for alpha in alphas:
            z = (1 - alpha) * mu1 + alpha * mu2
            decoded = model.decoder(z).cpu().view(28, 28)
            decoded_imgs.append(decoded)

    return decoded_imgs, alphas

In [ ]:
digit_a, digit_b = 3, 8
img_a = find_digit(test_dataset, digit_a)
img_b = find_digit(test_dataset, digit_b)

interp_imgs, alphas = interpolate_vae(vae_model, img_a, img_b, steps=12)

fig, axes = plt.subplots(1, len(interp_imgs), figsize=(len(interp_imgs) * 1.2, 1.8))
for i, (img, alpha) in enumerate(zip(interp_imgs, alphas)):
    axes[i].imshow(img.numpy(), cmap="gray")
    axes[i].axis("off")
    axes[i].set_title(f"{alpha:.1f}", fontsize=8)
fig.suptitle(f"VAE Latent Interpolation: {digit_a} -> {digit_b}", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
digit_a2, digit_b2 = 1, 7
img_a2 = find_digit(test_dataset, digit_a2)
img_b2 = find_digit(test_dataset, digit_b2)

interp_imgs2, alphas2 = interpolate_vae(vae_model, img_a2, img_b2, steps=12)

fig, axes = plt.subplots(1, len(interp_imgs2), figsize=(len(interp_imgs2) * 1.2, 1.8))
for i, (img, alpha) in enumerate(zip(interp_imgs2, alphas2)):
    axes[i].imshow(img.numpy(), cmap="gray")
    axes[i].axis("off")
    axes[i].set_title(f"{alpha:.1f}", fontsize=8)
fig.suptitle(f"VAE Latent Interpolation: {digit_a2} -> {digit_b2}", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 6: VQ-VAE (Vector Quantized VAE)

VQ-VAE replaces the continuous latent space with a **discrete codebook**.
The encoder outputs a continuous vector, which is then quantized to the
nearest codebook entry. Key concepts:

- **Straight-through estimator:** gradients pass through the quantization
  step unchanged (since argmin is not differentiable)
- **Commitment loss:** encourages the encoder output to stay close to the
  codebook entries it selects
- **Codebook loss:** moves codebook vectors toward the encoder outputs

Total loss = reconstruction + codebook_loss + beta * commitment_loss

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=64, embedding_dim=16, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1.0 / num_embeddings, 1.0 / num_embeddings)

    def forward(self, z_e):
        # z_e: (batch, embedding_dim)
        # Compute distances to all codebook entries
        distances = (
            torch.sum(z_e ** 2, dim=1, keepdim=True)
            + torch.sum(self.embeddings.weight ** 2, dim=1)
            - 2 * torch.matmul(z_e, self.embeddings.weight.t())
        )

        # Nearest codebook entry
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embeddings(encoding_indices)

        # Losses
        codebook_loss = F.mse_loss(z_q, z_e.detach())  # move codebook toward encoder
        commitment_loss = F.mse_loss(z_e, z_q.detach())  # move encoder toward codebook
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss

        # Straight-through estimator: copy gradients from z_q to z_e
        z_q_st = z_e + (z_q - z_e).detach()

        return z_q_st, vq_loss, encoding_indices

In [ ]:
class VQVAEEncoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, embedding_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 128)
        self.fc3 = nn.Linear(128, embedding_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc3(h)


class VQVAEDecoder(nn.Module):
    def __init__(self, embedding_dim=16, hidden_dim=256, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(embedding_dim, 128)
        self.fc2 = nn.Linear(128, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z_q):
        h = F.relu(self.fc1(z_q))
        h = F.relu(self.fc2(h))
        return torch.sigmoid(self.fc3(h))


class VQVAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, embedding_dim=16, num_embeddings=64):
        super().__init__()
        self.encoder = VQVAEEncoder(input_dim, hidden_dim, embedding_dim)
        self.quantizer = VectorQuantizer(num_embeddings, embedding_dim)
        self.decoder = VQVAEDecoder(embedding_dim, hidden_dim, input_dim)

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, indices = self.quantizer(z_e)
        x_recon = self.decoder(z_q)
        return x_recon, vq_loss, z_e, indices

In [ ]:
def train_vqvae(model, train_loader, epochs=20, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    history = {"total": [], "recon": [], "vq": []}

    for epoch in range(epochs):
        total_loss = 0.0
        total_recon = 0.0
        total_vq = 0.0

        for batch_imgs, _ in train_loader:
            batch_imgs = batch_imgs.view(-1, 784).to(device)
            x_recon, vq_loss, _, _ = model(batch_imgs)
            recon_loss = F.mse_loss(x_recon, batch_imgs)
            loss = recon_loss + vq_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_imgs.size(0)
            total_recon += recon_loss.item() * batch_imgs.size(0)
            total_vq += vq_loss.item() * batch_imgs.size(0)

        n = len(train_loader.dataset)
        history["total"].append(total_loss / n)
        history["recon"].append(total_recon / n)
        history["vq"].append(total_vq / n)

        if (epoch + 1) % 5 == 0:
            print(
                f"  Epoch {epoch+1}/{epochs} | "
                f"Total: {history['total'][-1]:.6f} | "
                f"Recon: {history['recon'][-1]:.6f} | "
                f"VQ: {history['vq'][-1]:.6f}"
            )

    return history

In [ ]:
vqvae_model = VQVAE(embedding_dim=16, num_embeddings=64).to(device)
print(f"VQ-VAE parameters: {sum(p.numel() for p in vqvae_model.parameters()):,}")
print(f"Codebook size: {vqvae_model.quantizer.num_embeddings} entries x {vqvae_model.quantizer.embedding_dim}D")
print("\nTraining VQ-VAE...")
vqvae_history = train_vqvae(vqvae_model, train_loader, epochs=20)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(vqvae_history["total"], linewidth=1.5)
axes[0].set_title("Total Loss")
axes[1].plot(vqvae_history["recon"], linewidth=1.5, color="tab:orange")
axes[1].set_title("Reconstruction (MSE)")
axes[2].plot(vqvae_history["vq"], linewidth=1.5, color="tab:green")
axes[2].set_title("VQ Loss (codebook + commitment)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
fig.suptitle("VQ-VAE Training Losses", fontsize=12)
plt.tight_layout()
plt.show()

### VQ-VAE Reconstructions

In [ ]:
def show_vqvae_reconstructions(model, data_loader, n=10, title="VQ-VAE Reconstructions"):
    model.eval()
    batch_imgs, _ = next(iter(data_loader))
    batch_imgs = batch_imgs[:n].to(device)
    with torch.no_grad():
        x_flat = batch_imgs.view(-1, 784)
        recon, _, _, _ = model(x_flat)
        recon = recon.view(-1, 1, 28, 28)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 2.8))
    for i in range(n):
        axes[0, i].imshow(batch_imgs[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Original", fontsize=10)
    axes[1, 0].set_ylabel("Recon", fontsize=10)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


show_vqvae_reconstructions(vqvae_model, test_loader)

### Codebook Utilization

A common issue with VQ-VAEs is codebook collapse, where only a fraction of
codebook entries are actually used. Let's check how many entries are being used.

In [ ]:
def check_codebook_usage(model, data_loader):
    model.eval()
    all_indices = []
    with torch.no_grad():
        for imgs, _ in data_loader:
            imgs = imgs.view(-1, 784).to(device)
            _, _, _, indices = model(imgs)
            all_indices.append(indices.cpu())

    all_indices = torch.cat(all_indices)
    unique_indices = torch.unique(all_indices)
    total = model.quantizer.num_embeddings

    print(f"Codebook utilization: {len(unique_indices)}/{total} entries used ({100*len(unique_indices)/total:.1f}%)")

    counts = torch.bincount(all_indices, minlength=total)
    plt.figure(figsize=(10, 3))
    plt.bar(range(total), counts.numpy(), width=1.0)
    plt.xlabel("Codebook Index")
    plt.ylabel("Usage Count")
    plt.title("VQ-VAE Codebook Entry Usage")
    plt.tight_layout()
    plt.show()

    return all_indices


vqvae_indices = check_codebook_usage(vqvae_model, test_loader)

---
## Part 7: Comparison — AE vs. VAE vs. VQ-VAE

We compare the three models on:
1. **Reconstruction quality** (visual + MSE)
2. **Latent space structure**
3. **Generation ability**

### 7.1 Reconstruction Quality (MSE on Test Set)

In [ ]:
def compute_test_mse(model, data_loader, model_type="ae"):
    model.eval()
    total_mse = 0.0
    total_samples = 0
    with torch.no_grad():
        for imgs, _ in data_loader:
            imgs_flat = imgs.view(-1, 784).to(device)
            if model_type == "ae":
                recon, _ = model(imgs_flat)
            elif model_type == "vae":
                recon, _, _, _ = model(imgs_flat)
            elif model_type == "vqvae":
                recon, _, _, _ = model(imgs_flat)
            mse = F.mse_loss(recon, imgs_flat, reduction="sum").item()
            total_mse += mse
            total_samples += imgs.size(0)
    return total_mse / total_samples


ae_mse = compute_test_mse(ae_model, test_loader, "ae")
vae_mse = compute_test_mse(vae_model, test_loader, "vae")
vqvae_mse = compute_test_mse(vqvae_model, test_loader, "vqvae")

print(f"Test MSE (per sample, summed over pixels):")
print(f"  Vanilla AE (2D latent):   {ae_mse:.2f}")
print(f"  VAE (2D latent):          {vae_mse:.2f}")
print(f"  VQ-VAE (16D, 64 codes):   {vqvae_mse:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
models_names = ["Vanilla AE\n(2D)", "VAE\n(2D)", "VQ-VAE\n(16D, 64 codes)"]
mse_values = [ae_mse, vae_mse, vqvae_mse]
colors = ["tab:blue", "tab:orange", "tab:green"]
bars = ax.bar(models_names, mse_values, color=colors, width=0.5)
for bar, val in zip(bars, mse_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.1f}", ha="center", fontsize=10)
ax.set_ylabel("MSE (per sample)")
ax.set_title("Reconstruction Quality on Test Set")
plt.tight_layout()
plt.show()

### 7.2 Side-by-Side Reconstructions

In [ ]:
n_compare = 8
batch_imgs, batch_labels = next(iter(test_loader))
batch_imgs = batch_imgs[:n_compare].to(device)

ae_model.eval()
vae_model.eval()
vqvae_model.eval()

with torch.no_grad():
    x_flat = batch_imgs.view(-1, 784)
    ae_recon, _ = ae_model(x_flat)
    vae_recon, _, _, _ = vae_model(x_flat)
    vqvae_recon, _, _, _ = vqvae_model(x_flat)

fig, axes = plt.subplots(4, n_compare, figsize=(n_compare * 1.3, 5.5))
row_labels = ["Original", "Vanilla AE", "VAE", "VQ-VAE"]
recons = [
    batch_imgs.cpu().view(-1, 28, 28),
    ae_recon.cpu().view(-1, 28, 28),
    vae_recon.cpu().view(-1, 28, 28),
    vqvae_recon.cpu().view(-1, 28, 28),
]

for row in range(4):
    for col in range(n_compare):
        axes[row, col].imshow(recons[row][col].numpy(), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(row_labels[row], fontsize=10, rotation=0, labelpad=55)

fig.suptitle("Reconstruction Comparison", fontsize=13)
plt.tight_layout()
plt.show()

### 7.3 Latent Space Structure Comparison

We compare the 2D latent spaces of the vanilla AE and VAE side by side.
The VQ-VAE has a 16D latent space with discrete codes, so we use t-SNE
to project its encoder outputs to 2D for visualization.

In [ ]:
from sklearn.manifold import TSNE

# Encode test set with VQ-VAE (use continuous encoder output for t-SNE)
vqvae_model.eval()
all_ze = []
all_vq_labels = []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.view(-1, 784).to(device)
        z_e = vqvae_model.encoder(imgs)
        all_ze.append(z_e.cpu())
        all_vq_labels.append(labels)

vqvae_ze = torch.cat(all_ze).numpy()
vqvae_labels_np = torch.cat(all_vq_labels).numpy()

# Subsample for t-SNE speed
n_tsne = 5000
rng = np.random.RandomState(42)
idx = rng.choice(len(vqvae_ze), size=n_tsne, replace=False)
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
vqvae_2d = tsne.fit_transform(vqvae_ze[idx])
vqvae_labels_sub = vqvae_labels_np[idx]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Vanilla AE
sc0 = axes[0].scatter(
    ae_z[:, 0].numpy(), ae_z[:, 1].numpy(),
    c=ae_labels.numpy(), cmap="tab10", s=2, alpha=0.5
)
axes[0].set_title("Vanilla AE (2D latent)")
axes[0].set_xlabel("z_0")
axes[0].set_ylabel("z_1")

# VAE
sc1 = axes[1].scatter(
    vae_z[:, 0].numpy(), vae_z[:, 1].numpy(),
    c=vae_labels.numpy(), cmap="tab10", s=2, alpha=0.5
)
axes[1].set_title("VAE (2D latent)")
axes[1].set_xlabel("z_0")
axes[1].set_ylabel("z_1")

# VQ-VAE (t-SNE)
sc2 = axes[2].scatter(
    vqvae_2d[:, 0], vqvae_2d[:, 1],
    c=vqvae_labels_sub, cmap="tab10", s=2, alpha=0.5
)
axes[2].set_title("VQ-VAE (16D, t-SNE projection)")
axes[2].set_xlabel("t-SNE 1")
axes[2].set_ylabel("t-SNE 2")

for ax in axes:
    cbar = plt.colorbar(ax.collections[0], ax=ax, ticks=range(10))
    cbar.set_label("Digit")

fig.suptitle("Latent Space Structure Comparison", fontsize=14)
plt.tight_layout()
plt.show()

### 7.4 Generation Ability

We test generation by sampling from each model's latent space:
- **Vanilla AE:** no principled way to sample; we try random points in the
  range of observed latent codes, but results are unpredictable.
- **VAE:** sample z ~ N(0, I) and decode -- works well by design.
- **VQ-VAE:** sample a random codebook entry and decode. Without a prior
  model (like a PixelCNN), the samples are random codebook decodings.

In [ ]:
n_samples = 10

# Vanilla AE: sample uniformly in the observed latent range
ae_model.eval()
z_min = ae_z.min(dim=0).values
z_max = ae_z.max(dim=0).values
with torch.no_grad():
    z_random_ae = z_min + (z_max - z_min) * torch.rand(n_samples, 2)
    ae_samples = ae_model.decoder(z_random_ae.to(device)).cpu().view(-1, 28, 28)

# VAE: sample from N(0, I)
vae_model.eval()
with torch.no_grad():
    z_random_vae = torch.randn(n_samples, 2).to(device)
    vae_samples = vae_model.decoder(z_random_vae).cpu().view(-1, 28, 28)

# VQ-VAE: sample random codebook entries
vqvae_model.eval()
with torch.no_grad():
    random_indices = torch.randint(0, vqvae_model.quantizer.num_embeddings, (n_samples,))
    z_random_vqvae = vqvae_model.quantizer.embeddings(random_indices).to(device)
    vqvae_samples = vqvae_model.decoder(z_random_vqvae).cpu().view(-1, 28, 28)

fig, axes = plt.subplots(3, n_samples, figsize=(n_samples * 1.3, 4.2))
row_labels = ["AE (random range)", "VAE (z~N(0,I))", "VQ-VAE (random code)"]
all_samples = [ae_samples, vae_samples, vqvae_samples]

for row in range(3):
    for col in range(n_samples):
        axes[row, col].imshow(all_samples[row][col].numpy(), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(row_labels[row], fontsize=9, rotation=0, labelpad=80)

fig.suptitle("Generation Comparison (Random Samples)", fontsize=13)
plt.tight_layout()
plt.show()

### Summary

| Property | Vanilla AE | VAE | VQ-VAE |
|---|---|---|---|
| **Latent space** | Continuous, unstructured | Continuous, regularized (Gaussian) | Discrete codebook |
| **Loss** | Reconstruction (MSE) | ELBO (Recon + KL) | Recon + codebook + commitment |
| **Reconstruction** | Good (but limited by 2D) | Slightly blurrier (KL tradeoff) | Sharp (higher-dim latent) |
| **Generation** | No principled sampling | Sample z ~ N(0,I) | Needs a prior model (e.g., PixelCNN) |
| **Latent structure** | Gaps, irregular | Smooth, continuous, complete | Discrete, clustered |
| **Key idea** | Compress and reconstruct | Probabilistic latent + reparameterization | Discrete bottleneck + straight-through |

**Key takeaways:**
- The VAE trades off some reconstruction quality for a well-structured latent
  space that enables principled generation.
- The VQ-VAE achieves sharp reconstructions with a discrete latent space, but
  requires a separate prior model (like PixelCNN or a transformer) to generate
  new samples -- the random codebook sampling shown above is not meaningful.
- The vanilla AE has no mechanism for generation; its latent space has no
  guaranteed structure.